# 02 — Pipeline de limpieza

**Objetivo:** documentar cada transformación aplicada al CSV crudo y verificar que el resultado es correcto antes de pasarlo al EDA.

**Conclusiones:**
- El CSV usa `;` como separador y contiene 1,251 filas × 9 columnas.
- Dos columnas de porcentaje (`Answer Rate`, `Service Level`) estaban como strings → convertidas a float [0, 1].
- Tres columnas de tiempo en formato `HH:MM:SS` → convertidas a segundos enteros.
- La columna `Index` es un row number redundante → eliminada.
- El CSV procesado se guarda en `data/processed/call_center_clean.csv`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Asegura que src/ esté en el path cuando se ejecuta desde notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from call_center.data_loader import load_raw, RAW_COLS, CLEAN_COLS
from call_center.cleaning import parse_percentage, parse_hhmmss_to_seconds, clean, save_processed

## 1. Carga del CSV crudo

In [2]:
raw = load_raw()
print(f"Shape: {raw.shape}")
raw.head()

Shape: (1251, 9)


,Index,Incoming Calls,Answered Calls,Answer Rate,Abandoned Calls,Answer Speed (AVG),Talk Duration (AVG),Waiting Time (AVG),Service Level (20 Seconds)
0,1,217,204,94.01%,13,0:00:17,0:02:14,0:02:45,76.28%
1,2,200,182,91.00%,18,0:00:20,0:02:22,0:06:55,72.73%
2,3,216,198,91.67%,18,0:00:18,0:02:38,0:03:50,74.30%
3,4,155,145,93.55%,10,0:00:15,0:02:29,0:03:12,79.61%
4,5,37,37,100.00%,0,0:00:03,0:02:06,0:00:35,97.30%


In [3]:
# Tipos originales — aquí se ve el problema
raw.dtypes

Index                         int64
Incoming Calls                int64
Answered Calls                int64
Answer Rate                     str
Abandoned Calls               int64
Answer Speed (AVG)              str
Talk Duration (AVG)             str
Waiting Time (AVG)              str
Service Level (20 Seconds)      str
dtype: object

In [4]:
# Nulos por columna
raw.isnull().sum()

Index                         0
Incoming Calls                0
Answered Calls                0
Answer Rate                   0
Abandoned Calls               0
Answer Speed (AVG)            0
Talk Duration (AVG)           0
Waiting Time (AVG)            0
Service Level (20 Seconds)    0
dtype: int64

## 2. Transformación de porcentajes

`"94.01%"` → `0.9401` (escala [0, 1])

In [5]:
# Antes
sample = raw[[RAW_COLS["answer_rate"], RAW_COLS["service_level"]]].head()
print("Antes:")
print(sample)
print(sample.dtypes)

# Después
print("\nDespués:")
print(parse_percentage(raw[RAW_COLS["answer_rate"]]).head())

Antes:
  Answer Rate Service Level (20 Seconds)
0      94.01%                     76.28%
1      91.00%                     72.73%
2      91.67%                     74.30%
3      93.55%                     79.61%
4     100.00%                     97.30%
Answer Rate                   str
Service Level (20 Seconds)    str
dtype: object

Después:
0    0.9401
1    0.9100
2    0.9167
3    0.9355
4    1.0000
Name: Answer Rate, dtype: float64


## 3. Transformación de tiempos HH:MM:SS → segundos

In [6]:
time_cols = [RAW_COLS["answer_speed"], RAW_COLS["talk_duration"], RAW_COLS["waiting_time"]]

print("Antes:")
print(raw[time_cols].head())

print("\nDespués (segundos):")
for col in time_cols:
    print(f"  {col}: {parse_hhmmss_to_seconds(raw[col]).head().tolist()}")

Antes:
  Answer Speed (AVG) Talk Duration (AVG) Waiting Time (AVG)
0            0:00:17             0:02:14            0:02:45
1            0:00:20             0:02:22            0:06:55
2            0:00:18             0:02:38            0:03:50
3            0:00:15             0:02:29            0:03:12
4            0:00:03             0:02:06            0:00:35

Después (segundos):
  Answer Speed (AVG): [17, 20, 18, 15, 3]
  Talk Duration (AVG): [134, 142, 158, 149, 126]
  Waiting Time (AVG): [165, 415, 230, 192, 35]


## 4. Pipeline completo

In [7]:
df = clean(raw)
print(f"Shape: {df.shape}")
df.dtypes

Shape: (1251, 8)


incoming_calls           int64
answered_calls           int64
answer_rate            float64
abandoned_calls          int64
answer_speed_avg_s       int64
talk_duration_avg_s      int64
waiting_time_avg_s       int64
service_level_20s      float64
dtype: object

In [8]:
df.head()

,incoming_calls,answered_calls,answer_rate,abandoned_calls,answer_speed_avg_s,talk_duration_avg_s,waiting_time_avg_s,service_level_20s
0,217,204,0.9401,13,17,134,165,0.7628
1,200,182,0.9100,18,20,142,415,0.7273
2,216,198,0.9167,18,18,158,230,0.7430
3,155,145,0.9355,10,15,149,192,0.7961
4,37,37,1.0000,0,3,126,35,0.9730


## 5. Estadísticas descriptivas del dataset limpio

In [9]:
df.describe().round(3)

,incoming_calls,answered_calls,answer_rate,abandoned_calls,answer_speed_avg_s,talk_duration_avg_s,waiting_time_avg_s,service_level_20s
count,1251.000,1251.000,1251.000,1251.000,1251.000,1251.000,1251.000,1251.000
mean,198.540,176.846,0.927,21.694,24.898,157.552,232.315,0.709
std,156.534,115.612,0.085,59.672,23.717,23.703,190.648,0.185
min,5.000,5.000,0.221,0.000,2.000,57.000,3.000,0.000
25%,123.000,114.000,0.914,3.000,13.000,142.000,118.000,0.602
50%,177.000,166.000,0.949,8.000,21.000,157.000,182.000,0.741
75%,233.000,214.500,0.972,16.000,30.000,171.000,276.000,0.841
max,1575.000,909.000,1.000,704.000,308.000,288.000,1551.000,1.000


## 6. Verificación de invariantes del negocio

Antes de guardar, validamos que los datos tienen sentido operacionalmente.

In [10]:
checks = {
    "Sin nulos": df.isnull().sum().sum() == 0,
    "answer_rate en [0,1]": df[CLEAN_COLS["answer_rate"]].between(0, 1).all(),
    "service_level en [0,1]": df[CLEAN_COLS["service_level"]].between(0, 1).all(),
    "answered <= incoming": (df[CLEAN_COLS["answered"]] <= df[CLEAN_COLS["incoming"]]).all(),
    "tiempos no negativos": (
        (df[CLEAN_COLS["answer_speed"]] >= 0) &
        (df[CLEAN_COLS["talk_duration"]] >= 0) &
        (df[CLEAN_COLS["waiting_time"]] >= 0)
    ).all(),
    "1251 filas": len(df) == 1251,
}

for check, result in checks.items():
    status = "OK" if result else "FALLO"
    print(f"  [{status}] {check}")

  [OK] Sin nulos
  [OK] answer_rate en [0,1]
  [OK] service_level en [0,1]
  [OK] answered <= incoming
  [OK] tiempos no negativos
  [OK] 1251 filas


## 7. Guardar CSV procesado

In [11]:
out = save_processed(df)
print(f"Guardado en: {out}")

Guardado en: C:\Users\PAOLA\Desktop\Proyectos personales\Call_Center\data\processed\call_center_clean.csv
